# Workflow Evaluation

Evaluation is critical for understanding how well your agent performs. This tutorial covers how to evaluate workflows using the NAT SDK.

## What You'll Learn

1. Understanding evaluation concepts
2. Creating evaluation datasets
3. Configuring evaluators (metrics)
4. Running evaluations via SDK
5. Running evaluations via CLI
6. Analyzing evaluation results

## Why Evaluate?

- **Measure performance** - Quantify how well your agent answers questions
- **Compare models** - Test different LLMs or prompts
- **Detect regressions** - Ensure changes don't break functionality
- **Benchmark** - Establish baseline metrics for improvement


In [1]:
import sys
from pathlib import Path

# Setup
module_path = Path("../../../src").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

print("✅ Environment configured")


✅ Environment configured


## Step 1: Create a Workflow to Evaluate

First, let's create a simple calculator workflow that we'll evaluate:


In [2]:
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.nim_llm import NimLLM
from nat.tool.datetime_tools import CurrentTimeTool
from nat.utils.sdk.nat_agent import NatAgent
from nat.utils.sdk.nat_function import NatFunction
from nat.utils.sdk.nat_function_group import NatFunctionGroup
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create LLM
llm = NimLLM(
    model_name="meta/llama-3.3-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create tools
tools: list[NatFunction | NatFunctionGroup | NatAgent] = []
time_tool = CurrentTimeTool(name="current_time")

# Import calculator if available
try:
    from nat_simple_calculator.register import CalculatorToolGroup
    calculator = CalculatorToolGroup(name="calculator")
    tools = [time_tool, calculator]
except ImportError:
    tools = [time_tool]
    print("⚠️  Calculator not installed. Install with:")
    print("   uv pip install -e examples/getting_started/simple_calculator")

# Create agent and workflow
agent = NatReActAgent(
    tools=tools,
    llm=llm,
    verbose=True,
)

workflow = NatWorkflow(entrypoint=agent)
print("✅ Workflow created")


/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Workflow created


## Step 2: Understanding Evaluation Datasets

Evaluation datasets contain:
- **id**: Unique identifier for each test case
- **question**: The question/prompt to send to the agent
- **answer**: The correct answer (ground truth)

NAT supports JSON and CSV dataset formats:

### JSON Format
```json
[
  {
    "id": "test_001",
    "question": "What is 2 + 2?",
    "answer": "The answer is 4."
  },
  {
    "id": "test_002",
    "question": "What is 10 * 5?",
    "answer": "The answer is 50."
  }
]
```

### CSV Format
```csv
id,question,answer
"test_001","What is 2 + 2?","The answer is 4."
"test_002","What is 10 * 5?","The answer is 50."
```

> **Note**: Answers should be descriptive strings that match expected agent responses, not just raw values.


## Step 3: Create an Evaluation Dataset

Let's create a sample evaluation dataset:


In [3]:
import json

# Create evaluation dataset with unique IDs for each test case
# Note: answers should be descriptive strings (matching expected agent responses)
eval_data = [
    {"id": "add_001", "question": "What is 2 + 2?", "answer": "The answer is 4."},
    {"id": "mul_001", "question": "What is 10 * 5?", "answer": "The answer is 50."},
    {"id": "div_001", "question": "What is 100 / 4?", "answer": "The answer is 25."},
    {"id": "sub_001", "question": "What is 15 - 7?", "answer": "The answer is 8."},
    {"id": "mixed_001", "question": "What is 3 * 3 + 1?", "answer": "The answer is 10."},
]

# Save to file
data_dir = Path("./data")
data_dir.mkdir(parents=True, exist_ok=True)

dataset_path = data_dir / "calculator_eval.json"
with open(dataset_path, "w") as f:
    json.dump(eval_data, f, indent=2)

print(f"📊 Evaluation dataset saved to: {dataset_path}")
print(f"   Contains {len(eval_data)} test cases")


📊 Evaluation dataset saved to: data/calculator_eval.json
   Contains 5 test cases


## Step 4: Configure Evaluators

Evaluators measure different aspects of your agent's performance. NAT provides several built-in evaluators:

| Evaluator | Measures | Use Case |
|-----------|----------|----------|
| **AnswerAccuracy** | Correctness of answers | General Q&A |
| **AnswerSimilarity** | Semantic similarity to expected | Flexible matching |
| **Faithfulness** | Groundedness in context | RAG applications |
| **ContextRelevancy** | Relevance of retrieved context | RAG applications |


In [4]:
from nat.eval.rag_evaluator.register import RagasEvaluator
from nat.utils.sdk.nat_evaluation import EvalDatasetJsonConfig
from nat.utils.sdk.nat_evaluation import NatEvaluation

# Create an accuracy evaluator
accuracy_evaluator = RagasEvaluator(
    llm=llm,                    # LLM used for evaluation
    metric="AnswerAccuracy",    # Metric to compute
    name="accuracy",            # Name for this evaluator
)

# Configure the evaluation
evaluation = NatEvaluation(
    output_dir=Path("./eval_results"),      # Where to save results
    dataset=EvalDatasetJsonConfig(          # Dataset configuration
        file_path=dataset_path,
    ),
    evaluators=[accuracy_evaluator],        # List of evaluators
)

# Add evaluation to workflow
workflow.add_evaluator(evaluation)

print("✅ Evaluation configured")


✅ Evaluation configured


## Step 5: Save the Configuration

Export the workflow with evaluation configuration:


In [5]:
# Save configuration
config_dir = Path("./configs")
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "eval_workflow.yaml"
workflow.save_to_config_file(config_path)

print(f"📄 Configuration saved to: {config_path}")
print("\n" + "=" * 60)
print("GENERATED CONFIGURATION WITH EVALUATION:")
print("=" * 60 + "\n")

with open(config_path) as f:
    print(f.read())


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


📄 Configuration saved to: configs/eval_workflow.yaml

GENERATED CONFIGURATION WITH EVALUATION:

functions:
  current_time:
    _type: current_datetime

function_groups:
  calculator:
    _type: calculator

llms:
  nim_llm:
    _type: nim
    model: meta/llama-3.3-70b-instruct
    max_tokens: 1024
    temperature: 0.0

workflow:
  _type: react_agent
  llm_name: nim_llm
  verbose: true
  tool_names:
  - current_time
  - calculator

eval:
  general:
    max_concurrency: 8
    workflow_alias: null
    output_dir: eval_results
    output: null
    dataset:
      _type: json
      file_path: data/calculator_eval.json
    profiler: null
  evaluators:
    accuracy:
      _type: ragas
      llm_name: nim_llm
      metric: AnswerAccuracy



## Step 6: Run Evaluation via Python

Run the evaluation directly in Python:


In [6]:
# Run evaluation (uncomment to execute)
await workflow.evaluate()
print("✅ Evaluation complete! Check ./eval_results for results")


Evaluating Ragas nv_accuracy: 100%|██████████| 5/5 [00:01<00:00,  3.30it/s]

✅ Evaluation complete! Check ./eval_results for results


## Step 7: Visualize Evaluation Results

Let's create a beautiful visualization of the evaluation results:


In [7]:
from utils.visualization import display_evaluation_dashboard

# Display the complete evaluation dashboard with a single function call!
# Configurable parameters:
#   - pass_threshold: Score required to pass (default: 0.8)
#   - warn_threshold: Score for warning vs fail (default: 0.5)
#   - show_summary/show_table/show_chart/show_reasoning: Toggle components
_ = display_evaluation_dashboard(
    eval_results_dir="./eval_results",
    pass_threshold=0.8,  # Customize as needed
)


📋 Detailed Test Results:



,ID,Question,Expected,Agent Response,Score,Status
0,add_001,What is 2 + 2?,The answer is 4.,4.0,100%,✅ Pass
1,mul_001,What is 10 * 5?,The answer is 50.,50.0,100%,✅ Pass
2,div_001,What is 100 / 4?,The answer is 25.,25.0,100%,✅ Pass
3,sub_001,What is 15 - 7?,The answer is 8.,8.0,100%,✅ Pass
4,mixed_001,What is 3 * 3 + 1?,The answer is 10.,10.0,100%,✅ Pass


In [8]:
# You can also display individual components with custom settings:
# Show only the summary with a stricter threshold
_ = display_evaluation_dashboard(
    eval_results_dir="./eval_results",
    pass_threshold=0.9,      # Stricter pass threshold
    warn_threshold=0.7,      # Adjust warning threshold
    show_table=False,        # Hide table
    show_chart=False,        # Hide chart
    show_reasoning=False,    # Hide reasoning
)


In [9]:
# The dashboard function returns the loaded data for further analysis:
data = display_evaluation_dashboard(
    eval_results_dir="./eval_results",
    show_summary=False,
    show_table=False,
    show_chart=True,  # Show just the chart
    show_reasoning=False,
)
if data:
    accuracy_data = data["accuracy_data"]
    print("\n📊 Custom Analysis:")
    print(f"   Average score: {accuracy_data['average_score']:.1%}")
    print(f"   Total tests: {len(accuracy_data['eval_output_items'])}")



📊 Custom Analysis:
   Average score: 100.0%
   Total tests: 5


In [10]:
# For low-level access to individual display functions:
# Load data manually if needed
import json

from utils.visualization import display_agent_reasoning

accuracy_file = Path("./eval_results/accuracy_output.json")
if accuracy_file.exists():
    with open(accuracy_file) as f:
        accuracy_data = json.load(f)
    # Display reasoning for a specific test case
    display_agent_reasoning(accuracy_data["eval_output_items"][0])


## Step 8: Run Evaluation via CLI

You can also run evaluation from the command line:

```bash
# Basic evaluation
nat eval --config_file configs/eval_workflow.yaml

# Evaluation with custom dataset
nat eval --config_file configs/eval_workflow.yaml \
    --dataset data/calculator_eval.json

# Skip running workflow (use cached results)
nat eval --config_file configs/eval_workflow.yaml --skip_workflow

# Run multiple times for statistical significance
nat eval --config_file configs/eval_workflow.yaml --reps 3
```


## Understanding Evaluation Results

After evaluation, you'll find results in the output directory:

```
eval_results/
├── eval_results.json       # Detailed results for each test case
├── eval_summary.json       # Aggregated metrics
└── eval_log.txt           # Execution log
```

### Example Results
```json
{
  "metrics": {
    "accuracy": {
      "mean": 0.85,
      "std": 0.12,
      "min": 0.6,
      "max": 1.0
    }
  },
  "test_cases": [
    {
      "id": "add_001",
      "question": "What is 2 + 2?",
      "answer": "The answer is 4.",
      "actual": "2 + 2 equals 4.",
      "accuracy": 1.0
    }
  ]
}
```


## Advanced: Multiple Evaluators

You can use multiple evaluators to get different perspectives:


In [11]:
# Example: Multiple evaluators
accuracy_eval = RagasEvaluator(llm=llm, metric="AnswerAccuracy", name="accuracy")
similarity_eval = RagasEvaluator(llm=llm, metric="AnswerSimilarity", name="similarity")

multi_evaluation = NatEvaluation(
    output_dir=Path("./eval_results_multi"),
    dataset=EvalDatasetJsonConfig(file_path=dataset_path),
    evaluators=[accuracy_eval, similarity_eval],
)

print("✅ Multi-evaluator configuration created")


✅ Multi-evaluator configuration created


## Summary

In this tutorial, you learned:

✅ How evaluation works in NAT  
✅ Creating evaluation datasets (JSON/CSV)  
✅ Configuring evaluators (RagasEvaluator)  
✅ Adding evaluation to workflows  
✅ Running evaluation via SDK and CLI  
✅ Understanding evaluation results  

## Next Steps

- **[10_profiling.ipynb](./10_profiling.ipynb)** - Measure latency and costs
- **[11_optimization.ipynb](./11_optimization.ipynb)** - Improve prompts and parameters
